# 实验六 · 生产者-消费者：有界缓冲区的四种实现

**所属**：《并行计算技术》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐⭐ 综合　|　**预计时长**：40–50 分钟

**对应讲义**：模块四（版本一、版本二）、模块五（版本三）、模块六（版本四）

> **实验说明**
> 1. 生产者-消费者是并发系统中最常见的结构：线程池的任务队列、网卡的收包环、日志系统的写入队列，本质都是它。本实验用四个版本逐步展开，每个版本解决一个新问题。
> 2. 本实验的四个版本各自独立成文件，**逐版本编写、编译、运行**，请按顺序执行，不要跳过中间版本——后一版本的设计动机来自前一版本暴露的问题。
> 3. 本实验横跨三个模块，**版本四属于模块六的内容**（涉及内存序），若讲义尚未讲到，可先跳过第 8 节，待实验九之后回看。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明有界缓冲区问题中的两类约束：**流量控制**与**互斥**，并指出它们各自对应的同步工具
- 使用两个计数信号量实现单生产者单消费者队列，并严格论证它为何**不需要互斥量**
- 说明多生产者多消费者场景下互斥量为何变得必需
- 掌握「**先取信号量，再取互斥量**」这一加锁顺序规则，并解释违反它为何导致死锁
- 使用条件变量实现同一问题，说明 `pthread_cond_wait` 的三步原子语义
- 解释等待条件为何必须写在 `while` 循环中，以及 `signal` 与 `broadcast` 的选择依据
- 使用原子索引实现无锁 SPSC 队列，说明其中的原子操作是为了**可见性**而非互斥
- 通过缓冲区大小的扫描，认识到阻塞与唤醒才是同步开销的主要来源

## 🗺️ 学习路径

1. **准备阶段**：理解有界缓冲区模型，区分流量控制与互斥这两类独立的约束
2. **版本一 · SPSC 信号量**：两个计数信号量即可，无需互斥量
   → 建立「信号量的计数值表达资源数量」这一认识
3. **版本二 · MPMC 信号量 + 互斥量**：多写者使索引成为临界区
   → 导出加锁顺序规则，与实验四的资源分级同源
4. **版本三 · 条件变量**：当条件比「计数」更复杂时的通用手段
   → 掌握 `while` 重检查、`signal` 与 `broadcast` 的选择
5. **版本四 · 无锁原子（进阶）**：用内存序取代锁
   → 认识原子操作在此处的作用是可见性而非互斥
6. **性能对比**：扫描缓冲区大小，找出真正主导开销的因素

## 1. 背景与动机

实验五中，线程之间传递的是**一条**消息，信号量的计数值始终在 0 与 1 之间——它退化成了一个「事件已发生」的标志。

本实验把场景扩展为**持续的数据流**：生产者不断产出数据项，消费者不断取走。此时会出现两个新问题：

1. **速率不匹配**。生产者可能快于消费者，也可能慢于消费者。缓冲区容量有限，满了怎么办？空了怎么办？
2. **多对多**。可能有多个生产者、多个消费者同时操作同一个缓冲区。

这一结构在实际系统中随处可见：

<!--
| 场景 | 生产者 | 消费者 | 缓冲区 |
|---|---|---|---|
| 线程池 | 提交任务的线程 | 工作线程 | 任务队列 |
| 网卡收包 | 网卡中断处理程序 | 协议栈线程 | 环形接收队列 |
| 日志系统 | 业务线程 | 写盘线程 | 日志缓冲区 |
| 流水线并行 | 上一级 | 下一级 | 级间队列 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">场景</th>
      <th style="text-align: left;">生产者</th>
      <th style="text-align: left;">消费者</th>
      <th style="text-align: left;">缓冲区</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">线程池</td>
      <td style="text-align: left;">提交任务的线程</td>
      <td style="text-align: left;">工作线程</td>
      <td style="text-align: left;">任务队列</td>
    </tr>
    <tr>
      <td style="text-align: left;">网卡收包</td>
      <td style="text-align: left;">网卡中断处理程序</td>
      <td style="text-align: left;">协议栈线程</td>
      <td style="text-align: left;">环形接收队列</td>
    </tr>
    <tr>
      <td style="text-align: left;">日志系统</td>
      <td style="text-align: left;">业务线程</td>
      <td style="text-align: left;">写盘线程</td>
      <td style="text-align: left;">日志缓冲区</td>
    </tr>
    <tr>
      <td style="text-align: left;">流水线并行</td>
      <td style="text-align: left;">上一级</td>
      <td style="text-align: left;">下一级</td>
      <td style="text-align: left;">级间队列</td>
    </tr>
  </tbody>
</table>

因此，掌握有界缓冲区的正确实现，是把本章前五个实验的知识组装成可用系统的关键一步。

## 2. 问题模型：有界缓冲区

一个容量为 $B$ 的**环形缓冲区**（ring buffer）。生产者不断放入数据项，消费者不断取出。

```
         in_index                        out_index
            ↓                                ↓
    ┌────┬────┬────┬────┬────┬────┬────┬────┐
    │    │    │ D3 │ D4 │ D5 │    │    │    │      B = 8
    └────┴────┴────┴────┴────┴────┴────┴────┘
      ▲                                     │
      └──────────── 首尾相接 ────────────────┘
```

- `in_index`：下一个写入位置，由生产者推进；
- `out_index`：下一个读取位置，由消费者推进；
- 二者都按 `(index + 1) % buffer_size` 环绕。

环形结构的好处是**空间复用**：数据被取走后槽位立即可以重新使用，无需搬移数据，也无需无限增长的内存。

## 3. 两类约束：流量控制与互斥

有界缓冲区必须同时满足两类约束。**它们是两个独立的问题，需要两种不同的工具**——这是本实验最重要的一条认识。

<!--
| 约束 | 要回答的问题 | 违反的后果 | 适用工具 |
|---|---|---|---|
| **流量控制** | 现在**有没有**可用资源？ | 缓冲区满时覆盖未读数据；空时读到无效值 | **信号量**（计数器天然表达资源数量） |
| **互斥** | **谁**能修改索引？ | 多个线程算出同一个索引，数据互相覆盖 | **互斥量** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">约束</th>
      <th style="text-align: left;">要回答的问题</th>
      <th style="text-align: left;">违反的后果</th>
      <th style="text-align: left;">适用工具</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>流量控制</strong></td>
      <td style="text-align: left;">现在<strong>有没有</strong>可用资源？</td>
      <td style="text-align: left;">缓冲区满时覆盖未读数据；空时读到无效值</td>
      <td style="text-align: left;"><strong>信号量</strong>（计数器天然表达资源数量）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>互斥</strong></td>
      <td style="text-align: left;"><strong>谁</strong>能修改索引？</td>
      <td style="text-align: left;">多个线程算出同一个索引，数据互相覆盖</td>
      <td style="text-align: left;"><strong>互斥量</strong></td>
    </tr>
  </tbody>
</table>

回顾前几个实验：

- **实验三**（π 估算）只有互斥问题——多个线程写同一个变量；
- **实验五**（消息传递）只有顺序问题——读必须发生在写之后；
- **本实验两者兼有**，因此需要把两种工具组合使用。

### 💡 信号量的计数能力

实验五中信号量的初值为 0，只用来表达「事件是否发生」。本实验将使用它的**计数**能力：

```c
sem_t empty_slots;   // 初值 = buffer_size，表示还有多少个空槽
sem_t full_slots;    // 初值 = 0，        表示还有多少个可读数据项
```

这两个信号量维持一个**不变式**：

$$\text{empty\_slots} + \text{full\_slots} + (\text{正在处理的项数}) = B$$

生产者取走一个空槽（`sem_wait(&empty_slots)`）、放入数据后发布一个数据项（`sem_post(&full_slots)`）；消费者反之。计数值天然地表达了「资源还剩多少」，这正是信号量的特点。

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys, os, re

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print("\n⚠️  当前仅 1 个核心：生产者与消费者只能分时轮转，无法真正并行，")
    print("    第 9 节测得的绝对耗时会偏高。相对趋势仍然成立。")
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### 编译与运行工具函数

本实验的四个版本各自独立成文件，逐一编译。信号量、条件变量与 C11 原子操作均由 `-lpthread` 链接。

In [2]:
SRC_DIR = "src_prodcons"
os.makedirs(SRC_DIR, exist_ok=True)


def compile_c(src, out):
    """用全章统一选项编译一个源文件，成功返回可执行文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", cmd)
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args, echo=True, timeout=300):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


def elapsed_ms(text):
    """从输出中提取总耗时（毫秒）。"""
    m = re.search(r"Total execution time:\s*([\d.]+)\s*ms", text)
    return float(m.group(1)) if m else None


## 5. 版本一 · SPSC：两个计数信号量

先考虑最简单的情形：**单生产者、单消费者**（Single-Producer Single-Consumer, SPSC）。

### 5.1 设计

```c
sem_t empty_slots;   // 初值 = buffer_size
sem_t full_slots;    // 初值 = 0
```

**生产者**：
```c
sem_wait(&empty_slots);              // P：取一个空槽，没有就阻塞
buffer[in_index] = item;
in_index = (in_index + 1) % buffer_size;
sem_post(&full_slots);               // V：发布一个数据项
```

**消费者**：
```c
sem_wait(&full_slots);               // P：取一个数据项，没有就阻塞
int item = buffer[out_index];
out_index = (out_index + 1) % buffer_size;
sem_post(&empty_slots);              // V：归还一个空槽
```

注意这两段代码的**对称性**：生产者消耗 `empty_slots` 而产出 `full_slots`，消费者恰好相反。流量控制由此自动完成——缓冲区满时生产者阻塞在 `sem_wait(&empty_slots)`，空时消费者阻塞在 `sem_wait(&full_slots)`。

In [ ]:
%%writefile {SRC_DIR}/pthread_producer_consumer_use_sem.c
#include <pthread.h>
#include <semaphore.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_BUFFER 1024

int buffer_size = 0;
int num_items = 0;

int* buffer = NULL;
int in_index = 0;   // written only by the producer
int out_index = 0;  // written only by the consumer

// Flow control only. empty_slots counts free slots, full_slots counts items.
// Their sum is invariant and equal to buffer_size.
sem_t empty_slots;
sem_t full_slots;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Single producer. No mutex is needed here: in_index has exactly one writer,
// and the two semaphores already guarantee that the slot being written has
// been released by the consumer.
void* Producer(void* arg) {
  (void)arg;
  for (int item = 0; item < num_items; ++item) {
    sem_wait(&empty_slots);  // take a free slot, block if none

    buffer[in_index] = item;
    if (num_items <= 20) {
      printf("Producer: produced %d at index %d\n", item, in_index);
    }
    in_index = (in_index + 1) % buffer_size;

    sem_post(&full_slots);  // publish the item
  }
  return NULL;
}

// Single consumer. Symmetrically, out_index has exactly one writer.
void* Consumer(void* arg) {
  (void)arg;
  for (int k = 0; k < num_items; ++k) {
    sem_wait(&full_slots);  // take an item, block if none

    int item = buffer[out_index];
    if (num_items <= 20) {
      printf("Consumer: consumed %d from index %d\n", item, out_index);
    }
    out_index = (out_index + 1) % buffer_size;

    sem_post(&empty_slots);  // release the slot
  }
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <buffer_size> <num_items>\n", argv[0]);
    return 1;
  }

  buffer_size = (int)strtol(argv[1], NULL, 10);
  num_items = (int)strtol(argv[2], NULL, 10);

  if (buffer_size <= 0 || buffer_size > MAX_BUFFER) {
    fprintf(stderr, "Error: buffer_size must be between 1 and %d\n",
            MAX_BUFFER);
    return 1;
  }
  if (num_items <= 0) {
    fprintf(stderr, "Error: num_items must be positive\n");
    return 1;
  }

  buffer = malloc(buffer_size * sizeof(int));
  if (buffer == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  printf("Producer-Consumer (SPSC, two counting semaphores)\n");
  printf("Buffer size: %d, Total items: %d\n\n", buffer_size, num_items);

  in_index = 0;
  out_index = 0;
  sem_init(&empty_slots, 0, buffer_size);  // all slots free at the start
  sem_init(&full_slots, 0, 0);             // no items at the start

  pthread_t producer_thread;
  pthread_t consumer_thread;

  double start = get_time_ms();
  if (pthread_create(&producer_thread, NULL, Producer, NULL) != 0 ||
      pthread_create(&consumer_thread, NULL, Consumer, NULL) != 0) {
    fprintf(stderr, "Error: pthread_create failed\n");
    return 1;
  }
  pthread_join(producer_thread, NULL);
  pthread_join(consumer_thread, NULL);
  double elapsed = get_time_ms() - start;

  printf("\nTotal execution time: %.3f ms\n", elapsed);

  sem_destroy(&empty_slots);
  sem_destroy(&full_slots);
  free(buffer);
  return 0;
}

In [ ]:
spsc = compile_c(f"{SRC_DIR}/pthread_producer_consumer_use_sem.c",
                 f"{SRC_DIR}/pthread_producer_consumer_use_sem")
print()
print("小规模演示：缓冲区 4，共 8 项（项数不超过 20 时程序会逐项打印）")
out = run_bin(spsc, 4, 8)

### 5.2 💡 为什么 SPSC 不需要互斥量

这是本实验最重要的一个判断。整个程序**一把锁都没有**，却是正确的。逐条对照实验三给出的数据竞争定义：

**① `in_index` 只有生产者写**，消费者从不碰它。只有一个写者，不构成竞争。

**② `out_index` 只有消费者写**，同理。

**③ `buffer[k]` 的访问被两个信号量在时间上完全隔开**：

- 生产者要写 `buffer[in_index]`，必须先 `sem_wait(&empty_slots)` 成功。这意味着该槽位已被消费者用 `sem_post(&empty_slots)` 归还——也就是说，消费者对该槽位的读取**已经完成**。
- 消费者要读 `buffer[out_index]`，必须先 `sem_wait(&full_slots)` 成功。这意味着生产者已 `sem_post(&full_slots)` 发布——也就是说，生产者对该槽位的写入**已经完成**。

因此，生产者与消费者**永远不会在同一时刻访问同一个槽位**。不满足数据竞争的条件一。

> **结论：在 SPSC 场景下加互斥量是多余的，只会平添开销。**

这与实验二的判据完全一致：需要保护的不是「共享数据」，而是「**被并发访问的同一个内存位置**」。信号量在这里同时完成了流量控制**与**访问隔离两件事。

从输出中也能看到流量控制在起作用：生产者填满 4 个槽位后被阻塞，必须等消费者取走一个才能继续。

## 6. 版本二 · MPMC：信号量 + 互斥量

### 6.1 多写者使索引成为临界区

一旦有**多个生产者**，情况立刻改变。关键在这一行：

```c
in_index = (in_index + 1) % buffer_size;
```

这是一个**读—改—写**，现在有多个写者。两个生产者可能读到同一个 `in_index`，写入同一个槽位，其中一个数据项就丢失了——这与实验三的 `global_sum += ...` 是同一类错误。

因此 MPMC 版本必须为索引加上互斥量：

```c
sem_wait(&empty_slots);              // 1. 先取资源

pthread_mutex_lock(&in_mutex);       // 2. 再进临界区
buffer[in_index] = item;
in_index = (in_index + 1) % buffer_size;
pthread_mutex_unlock(&in_mutex);

sem_post(&full_slots);               // 3. 发布
```

**生产者与消费者使用两把不同的互斥量**（`in_mutex` / `out_mutex`），因为它们修改的是不同的变量，没有理由互相阻塞。这是实验三「最小化临界区」规则的又一次应用。

In [ ]:
%%writefile {SRC_DIR}/pthread_producer_consumer_multi_use_sem.c
#include <pthread.h>
#include <semaphore.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_BUFFER 1024
#define MAX_THREADS 64

int buffer_size = 0;
int total_items = 0;
int producer_count = 0;
int consumer_count = 0;

int* buffer = NULL;
int in_index = 0;   // now written by EVERY producer
int out_index = 0;  // now written by EVERY consumer

// Flow control: how many slots / items are available.
sem_t empty_slots;
sem_t full_slots;

// Mutual exclusion: the indices now have several writers, so the
// read-modify-write on them is a critical section. Producers and consumers use
// separate mutexes because they touch different index variables.
pthread_mutex_t in_mutex;
pthread_mutex_t out_mutex;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Distributes total_items over thread_count threads as evenly as possible:
// the first (total_items % thread_count) threads take one extra item.
static int compute_my_items(long my_rank, int thread_count, int items) {
  int base = items / thread_count;
  int remainder = items % thread_count;
  return (my_rank < remainder) ? base + 1 : base;
}

// First item id owned by my_rank, so that produced ids stay globally unique.
static int compute_start_id(long my_rank, int thread_count, int items) {
  int base = items / thread_count;
  int remainder = items % thread_count;
  if (my_rank < remainder) {
    return (base + 1) * (int)my_rank;
  }
  return (base + 1) * remainder + base * ((int)my_rank - remainder);
}

void* Producer(void* rank) {
  long my_rank = (long)rank;
  int my_items = compute_my_items(my_rank, producer_count, total_items);
  int start_id = compute_start_id(my_rank, producer_count, total_items);

  for (int k = 0; k < my_items; ++k) {
    int item = start_id + k;

    // Order matters: take the semaphore FIRST, then the mutex. Blocking on
    // sem_wait while holding in_mutex would deadlock the system.
    sem_wait(&empty_slots);

    pthread_mutex_lock(&in_mutex);
    buffer[in_index] = item;
    if (total_items <= 20) {
      printf("Producer %ld: produced %d at index %d\n", my_rank, item,
             in_index);
    }
    in_index = (in_index + 1) % buffer_size;
    pthread_mutex_unlock(&in_mutex);

    sem_post(&full_slots);
  }
  return NULL;
}

void* Consumer(void* rank) {
  long my_rank = (long)rank;
  int my_items = compute_my_items(my_rank, consumer_count, total_items);

  for (int k = 0; k < my_items; ++k) {
    sem_wait(&full_slots);

    pthread_mutex_lock(&out_mutex);
    int item = buffer[out_index];
    if (total_items <= 20) {
      printf("Consumer %ld: consumed %d from index %d\n", my_rank, item,
             out_index);
    }
    out_index = (out_index + 1) % buffer_size;
    pthread_mutex_unlock(&out_mutex);

    sem_post(&empty_slots);
  }
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 5) {
    fprintf(stderr,
            "Usage: %s <buffer_size> <total_items> <producers> <consumers>\n",
            argv[0]);
    return 1;
  }

  buffer_size = (int)strtol(argv[1], NULL, 10);
  total_items = (int)strtol(argv[2], NULL, 10);
  producer_count = (int)strtol(argv[3], NULL, 10);
  consumer_count = (int)strtol(argv[4], NULL, 10);

  if (buffer_size <= 0 || buffer_size > MAX_BUFFER) {
    fprintf(stderr, "Error: buffer_size must be between 1 and %d\n",
            MAX_BUFFER);
    return 1;
  }
  if (total_items <= 0) {
    fprintf(stderr, "Error: total_items must be positive\n");
    return 1;
  }
  if (producer_count <= 0 || producer_count > MAX_THREADS ||
      consumer_count <= 0 || consumer_count > MAX_THREADS) {
    fprintf(stderr, "Error: producers and consumers must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  // Every consumer must be able to claim a share, otherwise some consumer
  // would wait forever on full_slots.
  if (total_items < producer_count || total_items < consumer_count) {
    fprintf(stderr, "Error: total_items must be >= producers and consumers\n");
    return 1;
  }

  buffer = malloc(buffer_size * sizeof(int));
  pthread_t* producers = malloc(producer_count * sizeof(pthread_t));
  pthread_t* consumers = malloc(consumer_count * sizeof(pthread_t));
  if (buffer == NULL || producers == NULL || consumers == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  printf("Producer-Consumer (MPMC, semaphores + mutex)\n");
  printf("Buffer: %d, Items: %d, Producers: %d, Consumers: %d\n\n", buffer_size,
         total_items, producer_count, consumer_count);

  in_index = 0;
  out_index = 0;
  sem_init(&empty_slots, 0, buffer_size);
  sem_init(&full_slots, 0, 0);
  pthread_mutex_init(&in_mutex, NULL);
  pthread_mutex_init(&out_mutex, NULL);

  double start = get_time_ms();
  for (long i = 0; i < producer_count; ++i) {
    if (pthread_create(&producers[i], NULL, Producer, (void*)i) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long i = 0; i < consumer_count; ++i) {
    if (pthread_create(&consumers[i], NULL, Consumer, (void*)i) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long i = 0; i < producer_count; ++i) pthread_join(producers[i], NULL);
  for (long i = 0; i < consumer_count; ++i) pthread_join(consumers[i], NULL);
  double elapsed = get_time_ms() - start;

  printf("\nTotal execution time: %.3f ms\n", elapsed);

  sem_destroy(&empty_slots);
  sem_destroy(&full_slots);
  pthread_mutex_destroy(&in_mutex);
  pthread_mutex_destroy(&out_mutex);
  free(buffer);
  free(producers);
  free(consumers);
  return 0;
}

In [ ]:
mpmc = compile_c(f"{SRC_DIR}/pthread_producer_consumer_multi_use_sem.c",
                 f"{SRC_DIR}/pthread_producer_consumer_multi_use_sem")
print()
print("小规模演示：缓冲区 4，共 12 项，3 个生产者，2 个消费者")
out = run_bin(mpmc, 4, 12, 3, 2)

### 6.2 为什么用两把互斥量

MPMC 版本没有用一把锁保护整个缓冲区，而是用了两把：

```c
pthread_mutex_t in_mutex;    // 保护 in_index，只有生产者使用
pthread_mutex_t out_mutex;   // 保护 out_index，只有消费者使用
```

原因是**生产者与消费者修改的是不同的变量**。生产者只推进 `in_index`，消费者只推进 `out_index`，二者之间不存在写冲突，因而没有理由互相阻塞。若合并为一把锁，生产者与消费者就会被迫排队，并发度降低一半。

这是实验三「最小化临界区」规则的又一次应用：**锁保护的是数据，而不是代码段**。既然是两组互不相关的数据，就应当用两把锁。

### 临界区内应当只保留必需的操作

再看临界区的范围：

```c
sem_wait(&empty_slots);              // 在临界区之外

pthread_mutex_lock(&in_mutex);
buffer[in_index] = item;             // 临界区内只有两件事：
in_index = (in_index + 1) % buffer_size;   // 写入数据、推进索引
pthread_mutex_unlock(&in_mutex);

sem_post(&full_slots);               // 在临界区之外
```

`sem_wait` 与 `sem_post` 都放在了临界区**之外**。这样安排有两点考虑：

1. **临界区应尽可能短**。信号量操作与索引更新无关，放进去只会延长其他线程的等待时间。
2. **应当避免在持有互斥量期间执行可能阻塞的操作**。`sem_wait` 在缓冲区满时会阻塞；若此时还持有互斥量，就把「等待外部事件」的时间计入了临界区，其他线程即使有事可做也只能干等。阻塞式 I/O、以及申请另一把可能被长期持有的锁，同样应当尽量移出临界区。

> 这是一条**工程建议**而非绝对规则——是否会造成实际问题，取决于具体的锁结构。
> 本实验的两把锁互不相关，即便把 `sem_wait` 移入临界区也不至于出错；但换一种锁结构，后果可能就大不相同。第 12 节的思考题会请你分析这样一种情形。

### 6.3 多生产者多消费者的任务划分

单生产者时，`for (item = 0; item < num_items; ++item)` 一句就够了。多生产者时则必须回答一个新问题：**这 `total_items` 项工作，如何分给 $m$ 个生产者？**

划分必须同时满足两个要求：

- **不重不漏**：各生产者产出的项数之和恰好等于 `total_items`；
- **编号唯一**：每一项有全局唯一的编号，便于在小规模演示中逐项核对是否被正确传递。

程序用两个辅助函数完成划分：

```c
static int compute_my_items(long my_rank, int thread_count, int items);   // 我做几项
static int compute_start_id(long my_rank, int thread_count, int items);   // 我从几号开始
```

划分规则是：设 `base = items / thread_count`、`remainder = items % thread_count`，则**前 `remainder` 个线程各做 `base + 1` 项，其余线程各做 `base` 项**。

#### 具体算例：`total_items = 10`，3 个生产者

此时 `base = 10 / 3 = 3`，`remainder = 10 % 3 = 1`，即第 0 号线程多做一项：

<!--
| 线程号 | 承担项数 | 起始编号 | 产出的编号 |
|---|---|---|---|
| 0 | `base + 1` = 4 | 0 | 0, 1, 2, 3 |
| 1 | `base` = 3 | 4 | 4, 5, 6 |
| 2 | `base` = 3 | 7 | 7, 8, 9 |
| **合计** | **10** | — | **0–9，不重不漏** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">线程号</th>
      <th style="text-align: left;">承担项数</th>
      <th style="text-align: left;">起始编号</th>
      <th style="text-align: left;">产出的编号</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">0</td>
      <td style="text-align: left;"><code>base + 1</code> = 4</td>
      <td style="text-align: left;">0</td>
      <td style="text-align: left;">0, 1, 2, 3</td>
    </tr>
    <tr>
      <td style="text-align: left;">1</td>
      <td style="text-align: left;"><code>base</code> = 3</td>
      <td style="text-align: left;">4</td>
      <td style="text-align: left;">4, 5, 6</td>
    </tr>
    <tr>
      <td style="text-align: left;">2</td>
      <td style="text-align: left;"><code>base</code> = 3</td>
      <td style="text-align: left;">7</td>
      <td style="text-align: left;">7, 8, 9</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>合计</strong></td>
      <td style="text-align: left;"><strong>10</strong></td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;"><strong>0–9，不重不漏</strong></td>
    </tr>
  </tbody>
</table>

消费者用**完全相同**的方式划分各自要取走的数量（消费者不需要起始编号，只需知道取几项）。

#### ⚠️ 为什么这一步不能出错

若生产总数与消费总数不相等，程序**必然挂起**，且两个方向都会挂：

<!--
| 情形 | 后果 |
|---|---|
| 生产总数 **<** 消费总数 | 多出的消费者阻塞在 `sem_wait(&full_slots)`，永远等不到数据项 |
| 生产总数 **>** 消费总数 | 缓冲区最终被填满，多出的生产者阻塞在 `sem_wait(&empty_slots)`，永远等不到空槽 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">情形</th>
      <th style="text-align: left;">后果</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">生产总数 <strong>&lt;</strong> 消费总数</td>
      <td style="text-align: left;">多出的消费者阻塞在 <code>sem_wait(&amp;full_slots)</code>，永远等不到数据项</td>
    </tr>
    <tr>
      <td style="text-align: left;">生产总数 <strong>&gt;</strong> 消费总数</td>
      <td style="text-align: left;">缓冲区最终被填满，多出的生产者阻塞在 <code>sem_wait(&amp;empty_slots)</code>，永远等不到空槽</td>
    </tr>
  </tbody>
</table>

> 这类挂起的表现与死锁完全相同（线程全部阻塞、CPU 占用为零），但**成因完全不同**——它不是同步机制的问题，而是**任务划分的算术错误**。
>
> 排查时可用实验四的方法：`gdb` 附加进程后执行 `thread apply all bt`。若线程停在 `sem_wait` 而非 `pthread_mutex_lock`，就应当首先核对生产与消费的总数是否相等。

真实系统中生产者的产出量往往事先未知，无法这样预先划分。届时需要另一种机制来通知消费者「不会再有数据了」——这一问题留作本实验的思考题。

## 7. 版本三 · 条件变量

### 7.1 信号量的表达能力边界

信号量的计数值只能表达「**资源还剩几个**」。若等待的条件更复杂，计数器就无能为力了，例如：

- 「队列长度超过一半时才唤醒批处理线程」
- 「队列非空**且**其中存在高优先级任务」
- 「所有工作线程都已到达某个阶段」（实验七的屏障）

**条件变量**（condition variable）可以等待**任意布尔条件**：

```c
pthread_mutex_t mutex;
pthread_cond_t not_empty;   // 等待队列：缓冲区变为非空
pthread_cond_t not_full;    // 等待队列：缓冲区变为非满
int count = 0;              // 真正的条件建立在这个普通变量上
```

> **核心认识：条件变量本身不保存任何状态，它只是一个等待队列。**
>
> 真正的条件是普通共享变量 `count`，由 `mutex` 保护。条件变量提供的只是「让线程睡在这里」和「把它们叫醒」两个动作。

**生产者**：
```c
pthread_mutex_lock(&mutex);
while (count == buffer_size) {              // 注意是 while，不是 if
  pthread_cond_wait(&not_full, &mutex);
}
buffer[in_index] = item;
in_index = (in_index + 1) % buffer_size;
++count;
pthread_cond_signal(&not_empty);
pthread_mutex_unlock(&mutex);
```

In [ ]:
%%writefile {SRC_DIR}/pthread_producer_consumer_use_cond.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_BUFFER 1024

int buffer_size = 0;
int num_items = 0;

int* buffer = NULL;
int in_index = 0;
int out_index = 0;
int count = 0;  // items currently in the buffer; the condition is built on it

// A condition variable stores no state of its own: it is only a wait queue.
// The actual condition is the ordinary variable 'count', protected by mutex.
pthread_mutex_t mutex;
pthread_cond_t not_empty;
pthread_cond_t not_full;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

void* Producer(void* arg) {
  (void)arg;
  for (int item = 0; item < num_items; ++item) {
    pthread_mutex_lock(&mutex);

    // while, not if: pthread_cond_wait may return spuriously, and another
    // thread may change count between the wakeup and the re-acquisition of
    // the mutex. The condition must be re-checked after every wakeup.
    while (count == buffer_size) {
      pthread_cond_wait(&not_full, &mutex);
    }

    buffer[in_index] = item;
    if (num_items <= 20) {
      printf("Producer: produced %d at index %d\n", item, in_index);
    }
    in_index = (in_index + 1) % buffer_size;
    ++count;

    pthread_cond_signal(&not_empty);
    pthread_mutex_unlock(&mutex);
  }
  return NULL;
}

void* Consumer(void* arg) {
  (void)arg;
  for (int k = 0; k < num_items; ++k) {
    pthread_mutex_lock(&mutex);

    while (count == 0) {
      pthread_cond_wait(&not_empty, &mutex);
    }

    int item = buffer[out_index];
    if (num_items <= 20) {
      printf("Consumer: consumed %d from index %d\n", item, out_index);
    }
    out_index = (out_index + 1) % buffer_size;
    --count;

    pthread_cond_signal(&not_full);
    pthread_mutex_unlock(&mutex);
  }
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <buffer_size> <num_items>\n", argv[0]);
    return 1;
  }

  buffer_size = (int)strtol(argv[1], NULL, 10);
  num_items = (int)strtol(argv[2], NULL, 10);

  if (buffer_size <= 0 || buffer_size > MAX_BUFFER) {
    fprintf(stderr, "Error: buffer_size must be between 1 and %d\n",
            MAX_BUFFER);
    return 1;
  }
  if (num_items <= 0) {
    fprintf(stderr, "Error: num_items must be positive\n");
    return 1;
  }

  buffer = malloc(buffer_size * sizeof(int));
  if (buffer == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  printf("Producer-Consumer (mutex + condition variables)\n");
  printf("Buffer size: %d, Total items: %d\n\n", buffer_size, num_items);

  in_index = 0;
  out_index = 0;
  count = 0;
  pthread_mutex_init(&mutex, NULL);
  pthread_cond_init(&not_empty, NULL);
  pthread_cond_init(&not_full, NULL);

  pthread_t producer_thread;
  pthread_t consumer_thread;

  double start = get_time_ms();
  if (pthread_create(&producer_thread, NULL, Producer, NULL) != 0 ||
      pthread_create(&consumer_thread, NULL, Consumer, NULL) != 0) {
    fprintf(stderr, "Error: pthread_create failed\n");
    return 1;
  }
  pthread_join(producer_thread, NULL);
  pthread_join(consumer_thread, NULL);
  double elapsed = get_time_ms() - start;

  printf("\nTotal execution time: %.3f ms\n", elapsed);

  pthread_mutex_destroy(&mutex);
  pthread_cond_destroy(&not_empty);
  pthread_cond_destroy(&not_full);
  free(buffer);
  return 0;
}

In [ ]:
cond = compile_c(f"{SRC_DIR}/pthread_producer_consumer_use_cond.c",
                 f"{SRC_DIR}/pthread_producer_consumer_use_cond")
print()
print("小规模演示：缓冲区 4，共 8 项")
out = run_bin(cond, 4, 8)

### 7.2 `pthread_cond_wait` 的三步原子语义

```c
int pthread_cond_wait(pthread_cond_t *cond, pthread_mutex_t *mutex);
```

调用它时，**原子地**完成三件事：

1. **释放 `mutex`**——否则其他线程无法进入临界区去改变条件，将永远死锁；
2. 把当前线程挂到 `cond` 的等待队列上并**进入睡眠**；
3. 被唤醒后，**重新获取 `mutex`**，然后才返回。

第 1、2 步必须**原子完成**，否则会出现这样的窗口：线程释放了锁但尚未挂到等待队列，此时另一线程恰好改变条件并发出信号——该信号找不到等待者而丢失，随后线程挂上队列，永远等不到下一次信号。这称为**丢失唤醒**（lost wakeup）。

这正是 `pthread_cond_wait` 必须同时接收 `cond` 和 `mutex` 两个参数的原因：只有它同时掌管二者，才能保证这一原子性。

> 这也是 6.2 节「持锁期间不做阻塞操作」规则的**唯一例外**：`pthread_cond_wait` 虽然阻塞，但它在阻塞前会先释放锁，因此不会造成死锁。

### 7.3 ⚠️ 为什么必须用 `while` 而不是 `if`

这是条件变量最常被写错的地方。三条理由，**任何一条都足以否决 `if`**：

**① 虚假唤醒（spurious wakeup）。** POSIX 标准明确允许 `pthread_cond_wait` 在没有任何 `signal` 的情况下返回。这不是实现缺陷，而是为了让实现能够高效（例如避免在信号处理时做额外同步）。

**② 唤醒与重新加锁之间存在窗口。** 线程 A 被唤醒后要重新竞争 `mutex`。在它获得锁之前，线程 B 可能抢先进入临界区并把条件重新破坏。A 获得锁时，条件可能已经不成立。

**③ `broadcast` 会唤醒所有等待者**，但资源可能只够一个线程使用。

> **规则**：`pthread_cond_wait` 返回**不代表条件成立**，只代表「值得再检查一次」。
> 因此必须放在 `while` 循环中，醒来后重新检查条件。

### 7.4 `signal` 与 `broadcast` 的选择

```c
int pthread_cond_signal(pthread_cond_t *cond);      // 唤醒至少一个等待者
int pthread_cond_broadcast(pthread_cond_t *cond);   // 唤醒全部等待者
```

判据很简单：

<!--
| 唤醒后的情形 | 应使用 |
|---|---|
| 只有**一个**线程能继续（如本实验：只多了一个数据项） | `signal` |
| **所有**线程都能继续（如屏障放行、读写锁释放写锁） | `broadcast` |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">唤醒后的情形</th>
      <th style="text-align: left;">应使用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">只有<strong>一个</strong>线程能继续（如本实验：只多了一个数据项）</td>
      <td style="text-align: left;"><code>signal</code></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>所有</strong>线程都能继续（如屏障放行、读写锁释放写锁）</td>
      <td style="text-align: left;"><code>broadcast</code></td>
    </tr>
  </tbody>
</table>

本实验每次只增加一个数据项或一个空槽，因此用 `signal`。若误用 `broadcast`，会唤醒全部等待者，而其中只有一个能成功，其余重新入睡——这称为**惊群效应**（thundering herd），是纯粹的浪费。

反过来，在需要 `broadcast` 的场合误用 `signal`，则会导致部分线程永远睡下去。实验七的屏障就是必须用 `broadcast` 的例子。

## 8. 版本四 · 无锁 SPSC（进阶 · 模块六）

> ⚠️ **本节涉及内存序（memory ordering），属于讲义模块六的内容。** 若尚未学习该模块，可先跳过本节，待实验九之后回看。

### 8.1 原子操作在此处是为了可见性，而非互斥

回到 SPSC 场景。版本一已经没有互斥量了，但两个索引仍是普通 `int`：

```c
int in_index = 0;    // 只有生产者写
int out_index = 0;   // 只有消费者写
```

在多核系统上，一个核心对 `in_index` 的写入**何时对另一个核心可见**，C 标准不作保证：编译器可能重排指令，处理器也可能乱序执行。版本四把索引改为 C11 原子类型：

```c
#include <stdatomic.h>

atomic_int in_index;    // 只有生产者写
atomic_int out_index;   // 只有消费者写

int idx = atomic_load(&in_index);
buffer[idx] = item;                                  // 先写数据
atomic_store(&in_index, (idx + 1) % buffer_size);    // 再发布索引
```

**这里的原子操作不是为了互斥**——每个索引仍然只有一个写者，不存在读—改—写竞争——**而是为了可见性与顺序**：

- `atomic_store` 默认采用顺序一致性，隐含**释放（release）语义**：它**前面**的所有写操作（包括 `buffer[idx] = item`）不得被重排到它**后面**；
- `atomic_load` 隐含**获取（acquire）语义**：它**后面**的所有读操作不得被重排到它**前面**。

因此，消费者一旦通过 `atomic_load` 观察到新的索引值，就**必然**能看到对应槽位中的数据。这个「**先写数据，再发布索引**」的配对，是所有无锁队列的基础。

> **`volatile` 不能替代原子操作。** 实验五说明了 `volatile` 只提供可见性、不提供原子性与内存序；
> 反过来，**原子变量已经隐含内存屏障，无需再加 `volatile`**。二者的职责不同，不可互换。

In [ ]:
%%writefile {SRC_DIR}/pthread_producer_consumer_use_sem_atomic.c
#include <pthread.h>
#include <semaphore.h>
#include <stdatomic.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define MAX_BUFFER 1024

int buffer_size = 0;
int num_items = 0;

int* buffer = NULL;

// Each index has exactly one writer, so there is no read-modify-write race.
// The atomics are here for visibility, not for mutual exclusion: they carry
// the memory barrier that publishes the buffer write to the other core.
atomic_int in_index;   // written only by the producer
atomic_int out_index;  // written only by the consumer

// Flow control only; no mutex anywhere in this program.
sem_t empty_slots;
sem_t full_slots;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

void* Producer(void* arg) {
  (void)arg;
  for (int item = 0; item < num_items; ++item) {
    sem_wait(&empty_slots);

    int idx = atomic_load(&in_index);
    buffer[idx] = item;
    if (num_items <= 20) {
      printf("Producer: produced %d at index %d\n", item, idx);
    }

    // Release semantics: the store to buffer[idx] above cannot be reordered
    // after this store, so a consumer that observes the new index is
    // guaranteed to observe the data as well.
    atomic_store(&in_index, (idx + 1) % buffer_size);

    sem_post(&full_slots);
  }
  return NULL;
}

void* Consumer(void* arg) {
  (void)arg;
  for (int k = 0; k < num_items; ++k) {
    sem_wait(&full_slots);

    int idx = atomic_load(&out_index);
    int item = buffer[idx];
    if (num_items <= 20) {
      printf("Consumer: consumed %d from index %d\n", item, idx);
    }

    atomic_store(&out_index, (idx + 1) % buffer_size);

    sem_post(&empty_slots);
  }
  return NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 3) {
    fprintf(stderr, "Usage: %s <buffer_size> <num_items>\n", argv[0]);
    return 1;
  }

  buffer_size = (int)strtol(argv[1], NULL, 10);
  num_items = (int)strtol(argv[2], NULL, 10);

  if (buffer_size <= 0 || buffer_size > MAX_BUFFER) {
    fprintf(stderr, "Error: buffer_size must be between 1 and %d\n",
            MAX_BUFFER);
    return 1;
  }
  if (num_items <= 0) {
    fprintf(stderr, "Error: num_items must be positive\n");
    return 1;
  }

  buffer = malloc(buffer_size * sizeof(int));
  if (buffer == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  printf("Producer-Consumer (lock-free SPSC, atomic indices)\n");
  printf("Buffer size: %d, Total items: %d\n\n", buffer_size, num_items);

  // atomic_init is not itself atomic: it must run before the threads start.
  atomic_init(&in_index, 0);
  atomic_init(&out_index, 0);
  sem_init(&empty_slots, 0, buffer_size);
  sem_init(&full_slots, 0, 0);

  pthread_t producer_thread;
  pthread_t consumer_thread;

  double start = get_time_ms();
  if (pthread_create(&producer_thread, NULL, Producer, NULL) != 0 ||
      pthread_create(&consumer_thread, NULL, Consumer, NULL) != 0) {
    fprintf(stderr, "Error: pthread_create failed\n");
    return 1;
  }
  pthread_join(producer_thread, NULL);
  pthread_join(consumer_thread, NULL);
  double elapsed = get_time_ms() - start;

  printf("\nTotal execution time: %.3f ms\n", elapsed);

  sem_destroy(&empty_slots);
  sem_destroy(&full_slots);
  free(buffer);
  return 0;
}

In [ ]:
atom = compile_c(f"{SRC_DIR}/pthread_producer_consumer_use_sem_atomic.c",
                 f"{SRC_DIR}/pthread_producer_consumer_use_sem_atomic")
print()
print("小规模演示：缓冲区 4，共 8 项")
out = run_bin(atom, 4, 8)

## 9. 性能对比：什么才是主要开销

四个版本的**数据传输量完全相同**，差别只在同步方式。下面先在固定缓冲区下比较，再扫描缓冲区大小以查明主导因素。

In [ ]:
import matplotlib.pyplot as plt

B, N, REP = 1024, 1000000, 3


def bench(binary, *args):
    """重复 REP 次取平均，而非取最优值。"""
    ts = [elapsed_ms(run_bin(binary, *args, echo=False)) for _ in range(REP)]
    return sum(ts) / len(ts)


rows = [
    ("SPSC 信号量", bench(spsc, B, N)),
    ("MPMC 信号量+锁", bench(mpmc, B, N, 1, 1)),
    ("条件变量", bench(cond, B, N)),
    ("无锁 SPSC 原子", bench(atom, B, N)),
]

print(f"缓冲区 {B}，数据项 {N:,}，重复 {REP} 次取平均，{os.cpu_count()} 核\n")
print(f"{'版本':<18}{'平均耗时(ms)':>14}{'相对最快':>12}")
print("-" * 46)
best = min(t for _, t in rows)
for name, t in rows:
    print(f"{name:<18}{t:>14.1f}{t / best:>11.2f}x")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = ["#295E96", "#E8833A", "#2E7D32", "#8E44AD"]
# 图中一律使用英文标签，避免依赖中文字体
labels_en = [
    "SPSC semaphore",
    "MPMC sem + mutex",
    "Condition variable",
    "Lock-free atomic",
]
bars = ax.bar(labels_en, [r[1] for r in rows], color=colors)
for b, (n, t) in zip(bars, rows):
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f"{t:.0f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylabel("Time (ms, lower is better)")
ax.set_title(f"Time of the four implementations "
             f"(buffer {B}, {N:,} items, {os.cpu_count()} cores)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 9.1 缓冲区大小的影响

上面的排序**并不稳定**——它取决于核心数、缓冲区大小以及生产与消费的速率是否匹配。下面扫描缓冲区大小，查明真正主导开销的因素。

In [ ]:
sizes = [1, 4, 16, 64, 256, 1024]
N2 = 50000
series = {"SPSC 信号量": [], "条件变量": [], "无锁 SPSC 原子": []}

print(f"数据项 {N2:,}\n")
print(f"{'缓冲区':>8}{'SPSC 信号量':>14}{'条件变量':>12}{'无锁原子':>12}")
print("-" * 48)
for b in sizes:
    t1 = elapsed_ms(run_bin(spsc, b, N2, echo=False))
    t2 = elapsed_ms(run_bin(cond, b, N2, echo=False))
    t3 = elapsed_ms(run_bin(atom, b, N2, echo=False))
    series["SPSC 信号量"].append(t1)
    series["条件变量"].append(t2)
    series["无锁 SPSC 原子"].append(t3)
    print(f"{b:>8}{t1:>14.1f}{t2:>12.1f}{t3:>12.1f}")

# 图例一律使用英文，避免依赖中文字体
LEGEND_EN = {
    "SPSC 信号量": "SPSC semaphore",
    "条件变量": "Condition variable",
    "无锁 SPSC 原子": "Lock-free atomic",
}

fig, ax = plt.subplots(figsize=(8.5, 4.4))
for (label, ys), c in zip(series.items(), ["#295E96", "#2E7D32", "#8E44AD"]):
    ax.plot(sizes, ys, "o-", label=LEGEND_EN[label], color=c, lw=2, ms=6)
ax.set_xscale("log", base=2)
ax.set_xticks(sizes)
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel("Buffer size (log scale)")
ax.set_ylabel("Time (ms, lower is better)")
ax.set_title(f"Effect of buffer size on synchronization overhead "
             f"({N2:,} items, {os.cpu_count()} cores)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

ratio = series["SPSC 信号量"][0] / series["SPSC 信号量"][-1]
print(f"\n缓冲区由 1 增至 1024，SPSC 信号量版本的耗时降低了 {ratio:.1f} 倍。")
print("三个版本呈现同样的趋势，说明主要开销并非来自同步原语本身，")
print("而是来自缓冲区满/空时的阻塞与唤醒——它意味着系统调用与上下文切换。")


### 9.2 结果解读

**① 缓冲区大小是首要因素。** 缓冲区越小，生产者与消费者越频繁地相互等待，每一次等待都意味着一次内核态阻塞与唤醒（`futex` 系统调用加上下文切换）。当缓冲区为 1 时，几乎每传递一个数据项都要付出一次这样的代价。

> **同步开销的主体不是加锁解锁，而是阻塞与唤醒。**
> 无竞争时，互斥量的加锁解锁只是一次用户态原子操作（数纳秒）；而一次阻塞唤醒需要数微秒。二者相差三个数量级。

**② 版本之间的排序取决于场景，没有固定的赢家。** 在缓冲区很小时，条件变量版本每传递一项都要经历完整的「加锁—检查—等待—唤醒—解锁」流程，开销最大；而在缓冲区充裕时，它的等待次数大幅减少，`pthread_cond_signal` 在无等待者时几乎无成本，反而可能领先。

**③ SPSC 信号量版与无锁原子版表现接近。** 二者都没有互斥量，差别仅在索引的发布方式。在单核或低竞争条件下差距不明显；核心数多、竞争激烈时，无锁版本通常更好，因为它避免了信号量的内核态挂起与唤醒。

**④ MPMC 版本即使只有一对线程，仍要付出互斥量的开销。** 这说明：**为通用性付出的代价，在特化场景下是纯粹的浪费**。

> 选择的依据永远是**负载特征**，而不是机制的先进程度。
> SPSC 场景就该用 SPSC 的解法；缓冲区该开多大，要用测量来决定。

### 关于本机结果

若在单核环境运行，生产者与消费者无法真正并行，绝对耗时会明显偏高，但上述**相对趋势依然成立**。请在鲲鹏多核平台上重跑本节，以获得有意义的绝对数值。

## 10. 结果分析

本实验把前五个实验的工具组装成了一个可用的并发结构，并建立了四项认识：

**① 两类约束需要两种工具。** 流量控制问的是「还有多少资源」，用信号量的计数值表达；互斥问的是「谁能改索引」，用互斥量表达。把二者混为一谈，就写不出正确的实现。

**② 同步的必要性取决于访问模式，而非线程数量。** SPSC 有两个线程却不需要互斥量，因为每个索引只有一个写者、且信号量已隔开了对槽位的访问；MPMC 需要，因为索引变成了多写者的读—改—写。判据仍是实验三给出的数据竞争定义。

**③ 组合使用多个同步对象时，顺序即规范。** 「先信号量、后互斥量」与实验四的资源分级是同一条规则。凡是需要同时持有多个同步对象的地方，都必须定义并遵守全局一致的获取顺序。

**④ 性能的主导因素是阻塞频率，而非原语的选择。** 缓冲区大小的影响远超版本之间的差异。优化并发程序时，应当首先设法**减少阻塞的次数**，而不是纠结于用哪一种锁。

### 🎓 结论

四个版本的能力与代价可以归纳为一张表：

<!--
| 版本 | 适用场景 | 同步对象 | 表达能力 | 代价 |
|---|---|---|---|---|
| **SPSC 信号量** | 单生产者单消费者 | 2 个信号量 | 资源计数 | 最小 |
| **MPMC 信号量+锁** | 多对多 | 2 信号量 + 2 互斥量 | 资源计数 + 互斥 | 中等 |
| **条件变量** | 条件复杂、非计数型 | 1 互斥量 + 2 条件变量 | **任意布尔条件** | 视场景而定 |
| **无锁原子** | 单生产者单消费者、低延迟 | 2 信号量 + 原子索引 | 资源计数 + 内存序 | 最小，但最难写对 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">适用场景</th>
      <th style="text-align: left;">同步对象</th>
      <th style="text-align: left;">表达能力</th>
      <th style="text-align: left;">代价</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>SPSC 信号量</strong></td>
      <td style="text-align: left;">单生产者单消费者</td>
      <td style="text-align: left;">2 个信号量</td>
      <td style="text-align: left;">资源计数</td>
      <td style="text-align: left;">最小</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>MPMC 信号量+锁</strong></td>
      <td style="text-align: left;">多对多</td>
      <td style="text-align: left;">2 信号量 + 2 互斥量</td>
      <td style="text-align: left;">资源计数 + 互斥</td>
      <td style="text-align: left;">中等</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>条件变量</strong></td>
      <td style="text-align: left;">条件复杂、非计数型</td>
      <td style="text-align: left;">1 互斥量 + 2 条件变量</td>
      <td style="text-align: left;"><strong>任意布尔条件</strong></td>
      <td style="text-align: left;">视场景而定</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>无锁原子</strong></td>
      <td style="text-align: left;">单生产者单消费者、低延迟</td>
      <td style="text-align: left;">2 信号量 + 原子索引</td>
      <td style="text-align: left;">资源计数 + 内存序</td>
      <td style="text-align: left;">最小，但最难写对</td>
    </tr>
  </tbody>
</table>

> 这四者不是「优劣排序」，而是**适用场景不同的四件工具**。
> 工程上的选择依据是：**先确定负载特征（几个生产者？几个消费者？等待条件是否可用计数表达？），再挑选表达能力恰好够用的那一个。**
>
> 表达能力过剩会带来不必要的开销（MPMC 用于 SPSC 场景），表达能力不足则根本写不对（用信号量表达复杂条件）。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察行为的变化：

1. 把 MPMC 版本中的 `sem_wait(&empty_slots)` 与 `pthread_mutex_lock(&in_mutex)` 调换顺序，并把 `in_mutex` 与 `out_mutex` 合并为一把锁，用小缓冲区运行（如 `2 100 4 4`）。观察程序是否挂起，再用 `gdb -p <pid>` 执行 `thread apply all bt`，指出死锁发生的位置。
2. 把条件变量版本中的 `while (count == buffer_size)` 改为 `if`，用缓冲区大小 1、1 个生产者、4 个消费者运行，尝试触发错误，并解释为什么多消费者更容易暴露这一缺陷。
3. 把条件变量版本的 `pthread_cond_signal` 全部改为 `pthread_cond_broadcast`，测量性能变化，并用「惊群效应」解释所得结果。
4. 把 MPMC 版本的 `in_mutex` 与 `out_mutex` 合并为一把锁，测量性能变化，解释为什么拆成两把更好。
5. 把无锁版本中的 `atomic_store` 改为 `atomic_store_explicit(..., memory_order_relaxed)`，运行并观察是否出错。若未出错，请说明为什么这仍然是错误的代码。
6. 用第 9.1 节的方法测量 MPMC 版本在不同生产者/消费者数量组合下的耗时（如 1×1、2×2、4×4），分析线程数与缓冲区大小之间的相互影响。

## 12. 🤔 思考题

- SPSC 版本没有使用互斥量，为什么是正确的？请逐条对照数据竞争的三个条件加以论证。
- 若把 SPSC 版本直接用于 2 个生产者，会出现什么错误？请指出具体是哪一行代码、哪一个变量出了问题。
- 条件变量版本中，`pthread_cond_signal` 是在 `pthread_mutex_unlock` 之前调用的。若移到 `unlock` 之后，程序是否仍然正确？两种写法各有什么优劣？
- 信号量可以用「互斥量 + 条件变量 + 一个计数器」实现。请写出伪代码。反过来，条件变量能否用信号量实现？若能，需要注意什么？
- 无锁 SPSC 队列中，`in_index` 与 `out_index` 分别只被一个线程写入。若二者恰好落在同一条缓存行上，会带来什么问题？应当如何解决？（提示：参见实验九。）
- 本实验的四个版本都要求生产总数与消费总数相等，否则程序挂起。真实系统中生产者的产出量往往事先未知，应当如何设计「结束通知」机制？请给出一种方案。
- 本实验的 MPMC 版本用了 `in_mutex` 与 `out_mutex` 两把互斥量。设想另一种常见写法：用**一把**互斥量保护整个缓冲区，并且把 `sem_wait(&empty_slots)` 移到 `pthread_mutex_lock` **之后**。请画出缓冲区已满时生产者与消费者各自的执行时序，判断程序会发生什么，并用实验四的 Coffman 四条件加以论证。若把两把锁的写法也照此调换顺序，结论是否相同？为什么？

## 13. 小结与后续

本实验用四个递进的版本，把本章前五个实验的工具组装成了一个完整的并发结构：

<!--
| 版本 | 解决的新问题 | 新增知识点 |
|---|---|---|
| **一 · SPSC 信号量** | 流量控制 | 信号量的计数能力、SPSC 为何无需互斥量 |
| **二 · MPMC 信号量+锁** | 多写者的索引竞争 | 加锁顺序规则、多线程负载划分 |
| **三 · 条件变量** | 非计数型的复杂条件 | 三步原子语义、`while` 重检查、`signal`/`broadcast` |
| **四 · 无锁原子** | 索引发布的可见性 | acquire/release 内存序、原子与 `volatile` 之别 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">解决的新问题</th>
      <th style="text-align: left;">新增知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>一 · SPSC 信号量</strong></td>
      <td style="text-align: left;">流量控制</td>
      <td style="text-align: left;">信号量的计数能力、SPSC 为何无需互斥量</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>二 · MPMC 信号量+锁</strong></td>
      <td style="text-align: left;">多写者的索引竞争</td>
      <td style="text-align: left;">加锁顺序规则、多线程负载划分</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>三 · 条件变量</strong></td>
      <td style="text-align: left;">非计数型的复杂条件</td>
      <td style="text-align: left;">三步原子语义、<code>while</code> 重检查、<code>signal</code>/<code>broadcast</code></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>四 · 无锁原子</strong></td>
      <td style="text-align: left;">索引发布的可见性</td>
      <td style="text-align: left;">acquire/release 内存序、原子与 <code>volatile</code> 之别</td>
    </tr>
  </tbody>
</table>

本章至此，同步工具已基本齐备：

<!--
| 需求 | 工具 | 引入实验 |
|---|---|---|
| 互斥 | 互斥量 | 实验三 |
| 多锁的安全获取 | 资源分级 | 实验四 |
| 等待单个事件 | 信号量（0/1） | 实验五 |
| 资源计数与流量控制 | 信号量（计数） | **本实验** |
| 等待任意条件 | 条件变量 | **本实验** |
| 可见性与内存序 | 原子操作 | **本实验（进阶）** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">需求</th>
      <th style="text-align: left;">工具</th>
      <th style="text-align: left;">引入实验</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">互斥</td>
      <td style="text-align: left;">互斥量</td>
      <td style="text-align: left;">实验三</td>
    </tr>
    <tr>
      <td style="text-align: left;">多锁的安全获取</td>
      <td style="text-align: left;">资源分级</td>
      <td style="text-align: left;">实验四</td>
    </tr>
    <tr>
      <td style="text-align: left;">等待单个事件</td>
      <td style="text-align: left;">信号量（0/1）</td>
      <td style="text-align: left;">实验五</td>
    </tr>
    <tr>
      <td style="text-align: left;">资源计数与流量控制</td>
      <td style="text-align: left;">信号量（计数）</td>
      <td style="text-align: left;"><strong>本实验</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">等待任意条件</td>
      <td style="text-align: left;">条件变量</td>
      <td style="text-align: left;"><strong>本实验</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">可见性与内存序</td>
      <td style="text-align: left;">原子操作</td>
      <td style="text-align: left;"><strong>本实验（进阶）</strong></td>
    </tr>
  </tbody>
</table>

➡️ **后续内容：实验七 向量归一化与屏障**。本实验的条件变量用于「一个线程等待另一个线程」。实验七将面对一类新的同步需求：**全体线程必须互相等待**——所有线程都到达某个点之后，任何一个才能继续。这就是**屏障**（barrier）。届时将看到，屏障可以用本实验的条件变量亲手实现，而这也正是 `pthread_cond_broadcast` 必须登场的场合。